In [ ]:
from pathlib import Path

# Establish robust, relative project root paths (notebooks sit 1 level below project root)
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
FIG_DIR, MODEL_DIR, METRIC_DIR = RESULTS / "figures", RESULTS / "models", RESULTS / "metrics"

# Ensure results directories exist
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)


# IMDB Dataset Audit & Exploratory Data Analysis (EDA)

**Course**: IT2011 — Artificial Intelligence and Machine Learning  
**Project**: K-Nearest Neighbors (KNN) Binary Sentiment Classification  
**Dataset**: IMDB Movie Review Dataset (`IMDB Dataset.csv`)  
**Stage**: Dataset Audit & Profiling (No Preprocessing / No Model Training)

---

### Important Notice
> **Strict Audit Protocol Enforced**:  
> In this stage, our sole objective is to audit, inspect, and understand the raw characteristics of the dataset.  
> **No text cleaning, stopword removal, stemming, lemmatization, duplicate dropping, row deletion, vectorization (TF-IDF / BoW), or model training is performed.**  
> The original dataset remains completely unaltered.


## 1. Import Necessary Libraries
We import standard data manipulation, statistical analysis, and visualization libraries taught in our university curriculum.


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 2. Load the Dataset Safely
We load the raw CSV file `IMDB Dataset.csv` from the resources folder and inspect its dimensions and in-memory footprint.


In [ ]:
# Define path to dataset
dataset_path = DATA_RAW / "IMDB Dataset.csv"

# Load raw dataset
df = pd.read_csv(dataset_path)

print(f"Dataset successfully loaded.")
print(f"Total Rows (Observations): {df.shape[0]:,}")
print(f"Total Columns (Variables): {df.shape[1]}")
print(f"In-Memory Usage: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")


## 3. Basic Dataset Inspection
Let's inspect the column names, data types, the first 5 rows, the last 5 rows, and a random sample of reviews.


In [ ]:
# Display DataFrame schema and types
print("=== Column Names and Data Types ===")
print(df.dtypes)
print("\n=== DataFrame Info Summary ===")
df.info()


In [ ]:
# Display First 5 Rows (Head)
print("=== First 5 Rows (Head) ===")
df.head()


In [ ]:
# Display Last 5 Rows (Tail)
print("=== Last 5 Rows (Tail) ===")
df.tail()


In [ ]:
# Display Random Sample of Reviews (Seed = 42)
print("=== Random Sample of 3 Reviews ===")
for idx, row in df.sample(3, random_state=42).iterrows():
    print(f"\n--- Review Index {idx} [{row['sentiment'].upper()}] ---")
    print(row['review'][:300] + "...")


## 4. Missing-Value Analysis
We perform an audit for:
1. Standard `NaN` / `null` values (`df.isnull().sum()`)
2. Empty strings (`len == 0`)
3. Whitespace-only strings (`str.strip() == ''`)


In [ ]:
# Missing value checks
missing_null = df.isnull().sum()
empty_str_review = (df['review'].str.len() == 0).sum()
whitespace_review = (df['review'].str.strip().str.len() == 0).sum()
empty_str_sentiment = (df['sentiment'].str.len() == 0).sum()
whitespace_sentiment = (df['sentiment'].str.strip().str.len() == 0).sum()

missing_audit = pd.DataFrame({
    'Column': ['review', 'sentiment'],
    'Null/NaN Count': [missing_null['review'], missing_null['sentiment']],
    'Empty String Count': [empty_str_review, empty_str_sentiment],
    'Whitespace-Only Count': [whitespace_review, whitespace_sentiment]
})

print("=== Missing Value Audit Results ===")
missing_audit


## 5. Duplicate Analysis
We check for exact duplicate rows (identical review and sentiment) and duplicate review texts.  
*Note: As per audit rules, we do not remove duplicates yet.*


In [ ]:
# Duplicate checks
exact_duplicates = df.duplicated().sum()
review_duplicates = df['review'].duplicated().sum()
unique_reviews = df['review'].nunique()
pct_duplicates = (exact_duplicates / len(df)) * 100

print(f"Exact Duplicate Rows: {exact_duplicates:,} ({pct_duplicates:.3f}%)")
print(f"Duplicate Review Texts: {review_duplicates:,} ({pct_duplicates:.3f}%)")
print(f"Unique Review Texts: {unique_reviews:,}")

# Check for conflicting labels (same review text with different sentiment)
conflicting = df.groupby('review')['sentiment'].nunique()
conflicting_count = (conflicting > 1).sum()
print(f"Reviews with Conflicting Labels: {conflicting_count}")

# Inspect a sample duplicate review
sample_dup_text = df[df.duplicated(keep=False)]['review'].iloc[0]
print(f"\nSample Duplicate Review Snippet: {sample_dup_text[:150]}...")
print("Occurrences of this review in dataset:")
print(df[df['review'] == sample_dup_text]['sentiment'].value_counts())


## 6. Sentiment / Class Distribution
We examine the distribution of the target variable `sentiment` to check for balance or imbalance.


In [ ]:
# Class counts and percentages
sentiment_counts = df['sentiment'].value_counts()
sentiment_pcts = df['sentiment'].value_counts(normalize=True) * 100

class_dist_df = pd.DataFrame({
    'Count': sentiment_counts,
    'Percentage (%)': sentiment_pcts
})

print("=== Class Distribution ===")
print(class_dist_df)


## 7. Review-Length Analysis
Without modifying the raw text, we compute descriptive statistics for:
- Character count (`len(text)`)
- Word count (`len(text.split())`)


In [ ]:
# Calculate character count and word count on raw text
df_len = df.copy()
df_len['char_count'] = df_len['review'].str.len()
df_len['word_count'] = df_len['review'].apply(lambda x: len(x.split()))

# Compute descriptive statistics
length_stats = pd.DataFrame({
    'Character Count': df_len['char_count'].describe(),
    'Word Count': df_len['word_count'].describe()
})

print("=== Review Length Descriptive Statistics ===")
length_stats.round(2)


In [ ]:
# Inspect Shortest and Longest Reviews
print("=== 5 Shortest Reviews (Word Count) ===")
for idx, r in df_len.nsmallest(5, 'word_count').iterrows():
    print(f"Index {idx} | Words: {r['word_count']} | Chars: {r['char_count']} | Sentiment: {r['sentiment']}")
    print(f"Text: '{r['review']}'\n")

print("=== 5 Longest Reviews (Word Count) ===")
for idx, r in df_len.nlargest(5, 'word_count').iterrows():
    print(f"Index {idx} | Words: {r['word_count']} | Chars: {r['char_count']} | Sentiment: {r['sentiment']}")
    print(f"Snippet: '{r['review'][:150]}...'\n")


## 8. Raw Text Inspection & Formatting Artifacts
We run regex-based profiling to quantify common textual noise, formatting artifacts, and linguistic phenomena across all 50,000 reviews.


In [ ]:
# Quantify raw text characteristics
raw_audit = {
    'Total Reviews': len(df),
    'Contains <br /> Tags': df['review'].str.contains(r'<br\s*/?>', regex=True).sum(),
    'Contains Any HTML Tag': df['review'].str.contains(r'<[a-zA-Z/][^>]*>', regex=True).sum(),
    'Contains Contractions (e.g., don\'t)': df['review'].str.contains(r"[a-zA-Z]+'[a-zA-Z]+", regex=True).sum(),
    'Contains Numbers/Digits': df['review'].str.contains(r'\d+', regex=True).sum(),
    'Contains ALL-CAPS Words (>=2 chars)': df['review'].str.contains(r'\b[A-Z]{2,}\b', regex=True).sum(),
    'Contains Repeated Chars (>=3 times)': df['review'].str.contains(r'(.)\1{2,}', regex=True).sum(),
    'Contains Ellipsis (...)': df['review'].str.contains(r'\.\.\.', regex=True).sum(),
    'Contains Exclamation Mark (!)': df['review'].str.contains(r'!', regex=True).sum(),
    'Contains Question Mark (?)': df['review'].str.contains(r'\?', regex=True).sum(),
    'Contains Non-ASCII Characters': df['review'].apply(lambda x: any(ord(c) >= 128 for c in x)).sum()
}

raw_audit_df = pd.DataFrame([
    {'Characteristic': k, 'Count': v, 'Percentage (%)': (v / len(df)) * 100}
    for k, v in raw_audit.items() if k != 'Total Reviews'
])

print("=== Raw Text Characteristics Audit ===")
raw_audit_df.round(2)


## 9. Visualizations
We generate exploratory charts to visualize:
1. Target class balance
2. Word count distributions across positive and negative sentiments
3. Boxplots showing median length and right-skewed outliers


In [ ]:
# Plot 1: Target Class Distribution
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='sentiment', hue='sentiment', palette=['#2ecc71', '#e74c3c'], legend=False, ax=ax)
ax.set_title("IMDB Dataset: Sentiment Class Distribution", fontsize=12, fontweight='bold')
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")

for p in ax.patches:
    height = int(p.get_height())
    ax.annotate(f"{height:,} ({height/len(df)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Review Word Count Distribution (Histogram & KDE)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df_len, x='word_count', hue='sentiment', bins=60, kde=True,
             palette=['#2ecc71', '#e74c3c'], element='step', ax=ax1)
ax1.set_xlim(0, 1200)
ax1.set_title("Word Count Distribution by Sentiment (0-1200 words)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Word Count")
ax1.set_ylabel("Frequency")

sns.boxplot(data=df_len, x='sentiment', y='word_count', hue='sentiment', palette=['#2ecc71', '#e74c3c'], legend=False, ax=ax2)
ax2.set_ylim(0, 1200)
ax2.set_title("Word Count Boxplot by Sentiment", fontsize=11, fontweight='bold')
ax2.set_xlabel("Sentiment")
ax2.set_ylabel("Word Count")

plt.tight_layout()
plt.show()


## 10. Summary of Findings

### Dataset Summary Table
| Metric | Value |
| :--- | :--- |
| **Total Rows** | `50,000` |
| **Total Columns** | `2` (`review`, `sentiment`) |
| **Input Feature** | `review` (Unstructured text) |
| **Target Variable** | `sentiment` (Binary categorical: `'positive'`, `'negative'`) |
| **Missing / Null Values** | `0` (100% complete) |
| **Duplicate Rows** | `418` (`0.836%`) |
| **Positive Reviews** | `25,000` (`50.0%`) |
| **Negative Reviews** | `25,000` (`50.0%`) |
| **Average Word Count** | `231.16` words |
| **Median Word Count** | `173.00` words |
| **Minimum Word Count** | `4` words |
| **Maximum Word Count** | `2,470` words |

### Key Takeaways for Next Steps:
1. **Perfect Class Symmetry**: Exactly 25,000 positive and 25,000 negative reviews. No imbalance mitigation needed.
2. **Complete Data Integrity**: Zero missing values or null records.
3. **Deduplication Required**: 418 duplicate records should be dropped during preprocessing to prevent train-test data leakage.
4. **Length is Non-Discriminative**: Word count medians for positive (172) and negative (174) reviews are nearly identical.
5. **Significant HTML Noise**: Over 58.4% of reviews contain `<br />` tags that must be stripped in preprocessing.
6. **High Skewness & Variance**: Review lengths range from 4 to 2,470 words, making length normalization ($L_2$ normalized TF-IDF) or Cosine distance essential for KNN.

> **Next Stage**: proceed to preprocessing design (Preprocessing Pipeline Design) only after user review.
